# Isolated BERTurk Embedding Experiment

This notebook implements the first embedding experiment for the thesis project. It evaluates whether word-level embeddings derived from BERTurk subtoken representations correspond to human semantic similarity judgments in the Turkish AnlamVer dataset.

The scope is deliberately narrow:

- input condition: isolated words only,
- model: `dbmdz/bert-base-turkish-cased` only,
- evaluated layers: 1, 7, and 12,
- pooling strategies: first subtoken, last subtoken, mean pooling, and max pooling.

The purpose is to validate the embedding extraction, subtoken pooling, caching, cosine similarity, and Spearman correlation pipeline before adding contextual sentence averaging or additional models.


## Methodological motivation

AnlamVer provides human similarity judgments for Turkish word pairs. If a model-derived word embedding captures semantic information in a way that aligns with human judgments, cosine similarities between model embeddings should correlate with the human similarity scores.

This first experiment uses isolated words because it is the simplest controlled condition: each target word is passed to BERTurk without sentence context. This makes the subtoken-to-word composition step easy to inspect and helps verify that the pipeline works before adding the more complex contextual condition.

Multiple layers are compared because BERT-like models distribute information across depth. Lower layers are often closer to lexical and form-based information, middle layers are often useful for transferable semantic information, and final layers can be more specialized to the pretraining objective. Comparing layers 1, 7, and 12 provides a compact first check of this layer effect.

Pooling strategies matter because Turkish words may be split into several subtokens. A word-level embedding must therefore be composed from multiple subtoken vectors. First and last pooling test whether one edge subtoken is sufficient, while mean and max pooling test whether information distributed across all subtokens gives a better representation.


In [ ]:
# Load libraries for data handling, model inference, and evaluation.
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr
from transformers import AutoModel, AutoTokenizer


pd.set_option("display.max_colwidth", 120)

# Use the current working directory as the project root, matching the first tokenization notebook.
PROJECT_ROOT = Path.cwd()
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "anlamver_tokenization_analysis.csv"
RESULTS_DIR = PROJECT_ROOT / "outputs" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_PATH = RESULTS_DIR / "0201-isolated_berturk_results.csv"
MODEL_NAME = "dbmdz/bert-base-turkish-cased"
LAYERS = [1, 7, 12]
POOLING_STRATEGIES = ["first", "middle", "last", "mean", "max"]

# Use the best available device for model inference.
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

#check check check
print(f"Project root: {PROJECT_ROOT}")
print(f"Data path: {DATA_PATH}")
print(f"Results output : {RESULTS_DIR}")
print(f"Device: {DEVICE}")

## Load and clean AnlamVer

The processed tokenization-analysis CSV is used as the input for this experiment. The thesis evaluation uses the two word columns (`W1`, `W2`) and the human semantic similarity score. If a `Sim_num` column already exists, it is used directly; otherwise it is created from `Sim`.


In [4]:
def numeric_similarity(series: pd.Series) -> pd.Series:
    """Convert similarity scores to float, accepting either dot or comma decimals."""
    # Accept both Turkish comma decimals and standard dot decimals.
    return pd.to_numeric(
        series.astype("string").str.replace(",", ".", regex=False),
        errors="coerce",
    )


# Load the processed AnlamVer tokenization file.
pairs = pd.read_csv(DATA_PATH)
required_columns = {"W1", "W2"}
missing_columns = required_columns - set(pairs.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

# Use Sim_num if it exists; otherwise create it from Sim.
if "Sim_num" not in pairs.columns:
    if "Sim" not in pairs.columns:
        raise ValueError("Expected either 'Sim_num' or 'Sim' in the dataset.")
    pairs["Sim_num"] = numeric_similarity(pairs["Sim"])
else:
    pairs["Sim_num"] = numeric_similarity(pairs["Sim_num"])

# Keep only complete word pairs with a valid similarity score.
pairs = pairs.dropna(subset=["W1", "W2", "Sim_num"]).copy()
pairs["W1"] = pairs["W1"].astype(str).str.strip()
pairs["W2"] = pairs["W2"].astype(str).str.strip()
pairs = pairs[(pairs["W1"] != "") & (pairs["W2"] != "")].reset_index(drop=True)

# Extract unique words so each word embedding is computed only once.
unique_words = sorted(pd.concat([pairs["W1"], pairs["W2"]]).dropna().unique())

print(f"Loaded {len(pairs):,} word pairs")
print(f"Unique words: {len(unique_words):,}")
pairs[["W1", "W2", "Sim_num"]].head()


Loaded 500 word pairs
Unique words: 317


,W1,W2,Sim_num
0,mızrap,barınak,0.166667
1,kırmızı,gül,1.166667
2,suçlu,şüphe,2.416667
3,laikçiler,sekülerizmciler,9.0
4,bitki,zeytin,3.666667


## Load BERTurk

The tokenizer and model are loaded from Hugging Face using `AutoTokenizer` and `AutoModel`. Hidden states are enabled because the experiment compares representations from layers 1, 7, and 12.


In [7]:
# Load BERTurk tokenizer and model with hidden states enabled.
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_hidden_states=True)

# Move the model to the selected device and use evaluation mode.
model.to(DEVICE)
model.eval()

print("*" * 80)
print(f"Loaded model '{MODEL_NAME}' on device '{DEVICE}'")
print(f"Tokenizer type: {type(tokenizer).__name__}")
print(f"Model type: {type(model).__name__}")
print(f"Number of hidden-state tensors, including embedding layer: {model.config.num_hidden_layers + 1}")


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8103.32it/s]
[transformers] BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


********************************************************************************
Loaded model 'dbmdz/bert-base-turkish-cased' on device 'mps'
Tokenizer type: BertTokenizer
Model type: BertModel
Number of hidden-state tensors, including embedding layer: 13


## Pooling functions

Each isolated word may map to one or more subtokens. The helper functions below reduce the subtoken matrix for a selected layer into one word-level vector.

The compared pooling strategies are `first`, `middle`, `last`, `mean`, and `max`. `middle` keeps the central subtoken representation. If a word has an even number of subtokens, the implementation uses the right-middle position because there is no single exact center.

In [8]:
def pool_first(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the first real subtoken as the word representation.
    return subtoken_embeddings[0]


def pool_middle(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the central real subtoken; for even counts this selects the right-middle subtoken.
    middle_index = len(subtoken_embeddings) // 2
    return subtoken_embeddings[middle_index]


def pool_last(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Use the last real subtoken as the word representation.
    return subtoken_embeddings[-1]


def pool_mean(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Average all real subtokens into one word vector.
    return subtoken_embeddings.mean(dim=0)


def pool_max(subtoken_embeddings: torch.Tensor) -> torch.Tensor:
    # Take the maximum value across subtokens for each vector dimension.
    return subtoken_embeddings.max(dim=0).values


# Map pooling names to reusable functions.
POOLING_FUNCTIONS = {
    "first": pool_first,
    "middle": pool_middle,
    "last": pool_last,
    "mean": pool_mean,
    "max": pool_max,
}

## Embedding extraction

For each word, the tokenizer first adds BERT special tokens such as `[CLS]` at the beginning and `[SEP]` at the end. These tokens are useful for BERT, but they are not part of the actual Turkish word. `get_non_special_token_indices` therefore returns the positions of only the real word subtokens.

For example, if the token sequence is `[CLS]`, `seküler`, `##izm`, `##ciler`, `[SEP]`, the non-special token indices are `1`, `2`, and `3`. These are the positions whose hidden-state vectors are pooled into one word-level embedding.

Layer numbers follow the usual BERT convention: layer 1 is the output of the first transformer block, and layer 12 is the final transformer layer. The embedding lookup layer would be index 0 and is not evaluated here.

In [9]:
def get_non_special_token_indices(input_ids: torch.Tensor) -> list[int]:
    """Return token positions excluding tokenizer special tokens such as [CLS] and [SEP]."""
    token_ids = input_ids[0].tolist()
    special_mask = tokenizer.get_special_tokens_mask(token_ids, already_has_special_tokens=True)
    # Keep only the positions that belong to the actual word.
    return [idx for idx, is_special in enumerate(special_mask) if not is_special]


def tokenize_word(word: str) -> list[str]:
    # Tokenize one isolated word and remove special tokens from the display.
    encoded = tokenizer(word, add_special_tokens=True, return_tensors="pt")
    token_ids = encoded["input_ids"][0].tolist()
    indices = get_non_special_token_indices(encoded["input_ids"])
    tokens = tokenizer.convert_ids_to_tokens(token_ids)
    return [tokens[idx] for idx in indices]


@torch.no_grad()
def extract_word_embeddings(word: str) -> dict[int, dict[str, np.ndarray]]:
    """Extract layer x pooling embeddings for one isolated word."""
    # Encode the word in isolation and send tensors to the model device.
    encoded = tokenizer(word, add_special_tokens=True, return_tensors="pt")
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    subtoken_indices = get_non_special_token_indices(encoded["input_ids"].cpu())
    if not subtoken_indices:
        raise ValueError(f"No non-special subtokens found for word: {word!r}")

    # Run BERTurk and retrieve all hidden layers.
    outputs = model(**encoded)
    hidden_states = outputs.hidden_states

    # Extract each selected layer and apply each pooling strategy.
    word_embeddings = {}
    for layer in LAYERS:
        layer_hidden = hidden_states[layer][0].detach().cpu()
        subtoken_embeddings = layer_hidden[subtoken_indices]
        word_embeddings[layer] = {
            pooling_name: pooling_fn(subtoken_embeddings).numpy()
            for pooling_name, pooling_fn in POOLING_FUNCTIONS.items()
        }
    return word_embeddings


## Validation checks

Before running the full dataset, this section checks that the pipeline behaves sensibly on a few small examples.

The first code block tokenizes several Turkish words and shows how many BERTurk subtokens each word becomes. This is a quick readability check: if a word is split, the later pooling functions need to combine those subtokens.

The second code block extracts embeddings for one example word and reports basic numeric diagnostics for every layer and pooling strategy:

- `shape` means the vector dimensions. For BERTurk base this should be `(768,)`, meaning one embedding vector with 768 numeric features.
- `finite_values` checks that all numbers are normal finite numbers. It should be `True`; `False` would mean at least one value is `NaN`, `inf`, or `-inf`.
- `l2_norm` is the Euclidean length of the vector. It is used here as a sanity check: the vector should not be all zeros, and its magnitude should look numerically reasonable.

The final validation block checks cosine similarity and structure. A word compared with itself should have cosine similarity very close to `1.0`, and the nested dictionary should contain every requested layer and pooling method.

In [10]:
# Inspect tokenization for a few example Turkish words.
example_words = ["mızrap", "kırmızı", "sekülerizmciler", "evlerimizden"]
example_tokenizations = pd.DataFrame(
    {"word": word, "berturk_subtokens": tokenize_word(word), "subtoken_count": len(tokenize_word(word))}
    for word in example_words
)
example_tokenizations


,word,berturk_subtokens,subtoken_count
0,mızrap,"[mız, ##rap]",2
1,kırmızı,[kırmızı],1
2,sekülerizmciler,"[sek, ##üler, ##izm, ##ciler]",4
3,evlerimizden,"[evleri, ##mi, ##z, ##den]",4


In [11]:
# Extract one example embedding to check vector dimensions and numeric values.
example_embedding = extract_word_embeddings("sekülerizmciler")
validation_rows = []
for layer, pooling_dict in example_embedding.items():
    for pooling_name, vector in pooling_dict.items():
        validation_rows.append(
            {
                "layer": layer,
                "pooling": pooling_name,
                "shape": vector.shape,
                "finite_values": bool(np.isfinite(vector).all()),
                "l2_norm": float(np.linalg.norm(vector)),
            }
        )

validation_table = pd.DataFrame(validation_rows)
validation_table

,layer,pooling,shape,finite_values,l2_norm
0,1,first,"(768,)",True,20.069136
1,1,middle,"(768,)",True,19.230291
2,1,last,"(768,)",True,18.763340
3,1,mean,"(768,)",True,13.309230
4,1,max,"(768,)",True,22.572126
5,7,first,"(768,)",True,16.888115
6,7,middle,"(768,)",True,16.781012
7,7,last,"(768,)",True,16.505344
8,7,mean,"(768,)",True,13.454658
9,7,max,"(768,)",True,18.785994


In [12]:
def cosine_similarity(vec_a: np.ndarray, vec_b: np.ndarray) -> float:
    """Compute cosine similarity with a small guard against zero vectors."""
    denom = np.linalg.norm(vec_a) * np.linalg.norm(vec_b)
    if denom == 0:
        return np.nan
    return float(np.dot(vec_a, vec_b) / denom)


# The same word should have cosine similarity very close to 1 with itself.
same_word_cosine = cosine_similarity(
    example_embedding[7]["mean"],
    extract_word_embeddings("sekülerizmciler")[7]["mean"],
)

# Check that the extraction pipeline produced the expected structure.
assert np.isclose(same_word_cosine, 1.0, atol=1e-6)
assert set(example_embedding.keys()) == set(LAYERS)
assert all(set(example_embedding[layer].keys()) == set(POOLING_STRATEGIES) for layer in LAYERS)
assert validation_table["finite_values"].all()

print(f"Same-word cosine sanity check: {same_word_cosine:.6f}")


Same-word cosine sanity check: 1.000000


## Build embedding cache

Embeddings are computed once per unique word and stored as `embeddings[word][layer][pooling]`. This avoids recomputing the same word for every pair and makes the pairwise evaluation deterministic and reproducible.


In [13]:
# Cache embeddings so repeated words are not recomputed for every pair.
embeddings: dict[str, dict[int, dict[str, np.ndarray]]] = {}

for index, word in enumerate(unique_words, start=1):
    embeddings[word] = extract_word_embeddings(word)
    if index % 50 == 0 or index == len(unique_words):
        print(f"Cached {index:>3}/{len(unique_words)} words")

print("Embedding cache complete")


Cached  50/317 words
Cached 100/317 words
Cached 150/317 words
Cached 200/317 words
Cached 250/317 words
Cached 300/317 words
Cached 317/317 words
Embedding cache complete


In [14]:
# Inspect a few cached entries to confirm the nested dictionary structure.
cache_shape_rows = []
for word in unique_words[:5]:
    for layer in LAYERS:
        cache_shape_rows.append(
            {
                "word": word,
                "layer": layer,
                "pooling_keys": sorted(embeddings[word][layer].keys()),
                "vector_shape": embeddings[word][layer]["mean"].shape,
            }
        )

pd.DataFrame(cache_shape_rows)


,word,layer,pooling_keys,vector_shape
0,adalet,1,"[first, last, max, mean, middle]","(768,)"
1,adalet,7,"[first, last, max, mean, middle]","(768,)"
2,adalet,12,"[first, last, max, mean, middle]","(768,)"
3,affedişi,1,"[first, last, max, mean, middle]","(768,)"
4,affedişi,7,"[first, last, max, mean, middle]","(768,)"
5,affedişi,12,"[first, last, max, mean, middle]","(768,)"
6,ahşap,1,"[first, last, max, mean, middle]","(768,)"
7,ahşap,7,"[first, last, max, mean, middle]","(768,)"
8,ahşap,12,"[first, last, max, mean, middle]","(768,)"
9,aile,1,"[first, last, max, mean, middle]","(768,)"


## Pairwise cosine similarities

For each word pair, the notebook retrieves the cached word embeddings and computes cosine similarity for every layer and pooling strategy.


In [15]:
def pair_similarity(row: pd.Series, layer: int, pooling: str) -> float:
    # Retrieve cached embeddings for both words in the pair.
    vec_w1 = embeddings[row["W1"]][layer][pooling]
    vec_w2 = embeddings[row["W2"]][layer][pooling]
    return cosine_similarity(vec_w1, vec_w2)


# Compute cosine similarity for each layer and pooling strategy.
similarity_columns = []
for layer in LAYERS:
    for pooling in POOLING_STRATEGIES:
        column = f"cosine_layer{layer}_{pooling}"
        pairs[column] = pairs.apply(pair_similarity, axis=1, layer=layer, pooling=pooling)
        similarity_columns.append(column)

pairs[["W1", "W2", "Sim_num", *similarity_columns[:4]]].head()


,W1,W2,Sim_num,cosine_layer1_first,cosine_layer1_middle,cosine_layer1_last,cosine_layer1_mean
0,mızrap,barınak,0.166667,0.297542,0.248598,0.248598,0.350366
1,kırmızı,gül,1.166667,0.344896,0.344896,0.344896,0.344896
2,suçlu,şüphe,2.416667,0.369548,0.369548,0.369548,0.369548
3,laikçiler,sekülerizmciler,9.0,0.369945,0.368383,0.654039,0.614763
4,bitki,zeytin,3.666667,0.448708,0.448708,0.448708,0.448708


## Spearman evaluation

Spearman correlation is used because the evaluation asks whether model similarities preserve the rank ordering of human similarity judgments. This is standard for intrinsic word similarity benchmarks and does not assume a linear relationship between cosine similarity and human scores.


In [16]:
# Correlate model cosine similarities with human similarity scores.
results = []
for layer in LAYERS:
    for pooling in POOLING_STRATEGIES:
        column = f"cosine_layer{layer}_{pooling}"
        valid = pairs[["Sim_num", column]].dropna()
        rho, p_value = spearmanr(valid["Sim_num"], valid[column])
        results.append(
            {
                "layer": layer,
                "pooling": pooling,
                "spearman_rho": rho,
                "n_pairs": len(valid),
            }
        )

# Sort the final table by layer and pooling name.
results_df = pd.DataFrame(results).sort_values(["layer", "pooling"]).reset_index(drop=True)
results_df


,layer,pooling,spearman_rho,n_pairs
0,1,first,0.426694,500
1,1,last,0.333949,500
2,1,max,0.232726,500
3,1,mean,0.325226,500
4,1,middle,0.317136,500
5,7,first,0.315051,500
6,7,last,0.287228,500
7,7,max,0.280261,500
8,7,mean,0.305104,500
9,7,middle,0.275134,500


## Save results

The final table contains one row for each layer-by-pooling combination. This is the main output of the first isolated-word BERTurk embedding experiment.


In [19]:
# Save the final result table for later thesis analysis.
results_df.to_csv(RESULTS_PATH, index=False)
print(f"Saved results to: {RESULTS_PATH}")
results_df.sort_values("spearman_rho", ascending=False)


Saved results to: /Users/nebox/Library/CloudStorage/GoogleDrive-anebieren@gmail.com/My Drive/ORTAK/nebox/Lectures/+2nd semester/Thesis/+thesis-proJect/outputs/results/0201-isolated_berturk_results.csv


,layer,pooling,spearman_rho,n_pairs
0,1,first,0.426694,500
13,12,mean,0.356264,500
10,12,first,0.356035,500
11,12,last,0.340645,500
1,1,last,0.333949,500
3,1,mean,0.325226,500
12,12,max,0.322338,500
14,12,middle,0.321913,500
4,1,middle,0.317136,500
5,7,first,0.315051,500


## Interpretation placeholder

This notebook is primarily a pipeline-validation experiment. The final correlations should be interpreted as the first isolated-word BERTurk baseline, not as the full thesis result. Later notebooks can add contextual sentence averaging, additional models, fragmentation analyses, and statistical comparisons once this extraction and pooling pipeline has been verified.
